# Exp 1: Activation Sparsity Analysis — Testing ViT-5's "Over-Gating" Claim

ViT-5 (arXiv 2602.08071) claims that combining LayerScale + SwiGLU causes "over-gating" —
excessive activation sparsity that degrades performance. However, **they never actually measure
the sparsity**. This notebook does exactly that.

We measure at every FFN layer:
- **Hoyer sparsity** (L1/L2 ratio normalized to [0,1])
- **Gini coefficient** (inequality of activation magnitudes)
- **% near-zero activations** (fraction of activations with |x| < 1e-3)

Models compared:
1. **DeiT-III-Small** (vanilla ViT baseline)
2. **ViT-5-Small** (all 7 components)

**Memory-efficient**: metrics computed on-the-fly, no raw tensors stored. Runs on T4 16GB.

In [ ]:
!pip install -q timm einops huggingface_hub matplotlib

In [ ]:
!git clone https://github.com/wangf3014/ViT-5.git vit5_repo 2>/dev/null || echo 'Already cloned'

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs('checkpoints', exist_ok=True)
vit5_small_ckpt = hf_hub_download(
    repo_id='FengWang3211/ViT-5',
    filename='vit5_small_patch16_224.pth',
    local_dir='checkpoints'
)
print(f'Downloaded: {vit5_small_ckpt}')

In [ ]:
import sys
sys.path.insert(0, 'vit5_repo')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# Load models ONE AT A TIME to save VRAM
# ============================================================

from models_vit5 import vit5_small, deit_small_patch16_LS
import timm

def load_vit5_small():
    model = vit5_small(img_size=224)
    ckpt = torch.load(vit5_small_ckpt, map_location='cpu', weights_only=False)
    state_dict = ckpt['model'] if 'model' in ckpt else ckpt
    model.load_state_dict(state_dict, strict=False)
    return model.to(device).eval()

def load_deit3_small():
    model = timm.create_model('deit3_small_patch16_224.fb_in1k', pretrained=True)
    return model.to(device).eval()

print('Model loaders ready.')

In [ ]:
# ============================================================
# Data: CIFAR-100 (auto-downloads, no ImageNet needed)
# Sparsity patterns are architecture-dependent, not data-dependent.
# ============================================================

from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

IMAGENET_VAL = '/content/imagenet/val'
if os.path.exists(IMAGENET_VAL):
    print('Using ImageNet validation set')
    dataset = datasets.ImageFolder(IMAGENET_VAL, transform=transform)
else:
    print('Using CIFAR-100 as proxy (sparsity is architecture-dependent).')
    dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

NUM_SAMPLES = 500
subset = torch.utils.data.Subset(dataset, list(range(min(NUM_SAMPLES, len(dataset)))))
loader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False, num_workers=2)
print(f'{len(subset)} samples, batch_size=32, {len(loader)} batches')

In [ ]:
# ============================================================
# ONLINE sparsity metrics — no raw tensors stored
# ============================================================

class OnlineSparsityTracker:
    """Computes sparsity metrics incrementally without storing activations.
    
    For each layer, tracks running sums needed for:
    - Hoyer sparsity (needs L1, L2 per sample)
    - % near-zero
    - Channel max (for dead channel detection)
    """
    
    def __init__(self, num_layers):
        self.num_layers = num_layers
        self.reset()
    
    def reset(self):
        self.hoyer_sum = [0.0] * self.num_layers
        self.gini_sum = [0.0] * self.num_layers
        self.near_zero_sum = [0.0] * self.num_layers
        self.total_elements = [0] * self.num_layers
        self.channel_max = [None] * self.num_layers  # track max |activation| per channel
        self.count = [0] * self.num_layers  # number of samples processed
    
    def update(self, layer_idx, activation):
        """Update metrics with a batch of activations. activation: (B, N, D) on GPU."""
        with torch.no_grad():
            B, N, D = activation.shape
            act = activation.float()
            
            # --- Hoyer sparsity (per-sample, averaged) ---
            flat = act.reshape(B, -1)  # (B, N*D)
            n = flat.shape[1]
            l1 = flat.abs().sum(dim=1)  # (B,)
            l2 = flat.norm(p=2, dim=1)  # (B,)
            sqrt_n = np.sqrt(n)
            hoyer = (sqrt_n - l1 / (l2 + 1e-12)) / (sqrt_n - 1 + 1e-12)
            self.hoyer_sum[layer_idx] += hoyer.clamp(0, 1).sum().item()
            
            # --- Gini coefficient (per-sample, averaged) ---
            abs_flat = flat.abs()
            sorted_x, _ = torch.sort(abs_flat, dim=1)
            indices = torch.arange(1, n + 1, device=act.device, dtype=torch.float32)
            numerator = (2 * (indices * sorted_x).sum(dim=1))
            denominator = n * sorted_x.sum(dim=1) + 1e-12
            gini = numerator / denominator - (n + 1) / n
            self.gini_sum[layer_idx] += gini.sum().item()
            
            # --- % near zero ---
            self.near_zero_sum[layer_idx] += (act.abs() < 1e-3).float().sum().item()
            self.total_elements[layer_idx] += act.numel()
            
            # --- Channel max (for dead channel detection) ---
            batch_ch_max = act.abs().reshape(-1, D).max(dim=0).values  # (D,)
            if self.channel_max[layer_idx] is None:
                self.channel_max[layer_idx] = batch_ch_max
            else:
                self.channel_max[layer_idx] = torch.max(self.channel_max[layer_idx], batch_ch_max)
            
            self.count[layer_idx] += B
    
    def get_metrics(self):
        metrics = {}
        for i in range(self.num_layers):
            if self.count[i] == 0:
                continue
            ch_max = self.channel_max[i]
            metrics[i] = {
                'hoyer': self.hoyer_sum[i] / self.count[i],
                'gini': self.gini_sum[i] / self.count[i],
                'pct_near_zero': 100.0 * self.near_zero_sum[i] / self.total_elements[i],
                'pct_dead_channels': 100.0 * (ch_max < 1e-3).float().mean().item() if ch_max is not None else 0,
            }
        return metrics

print('OnlineSparsityTracker ready.')

In [ ]:
# ============================================================
# Run analysis for a model — hooks compute metrics on GPU,
# then immediately discard the activation tensor.
# Peak VRAM = model weights + 1 batch of activations.
# ============================================================

def analyze_model(model, loader, model_name):
    """Run forward passes, compute sparsity metrics online."""
    num_layers = len(model.blocks)
    tracker = OnlineSparsityTracker(num_layers)
    hooks = []
    
    # Hook on post-activation (after GELU)
    for i, block in enumerate(model.blocks):
        def make_hook(idx):
            def hook_fn(module, input, output):
                tracker.update(idx, output)  # compute on GPU, no storage
            return hook_fn
        hooks.append(block.mlp.act.register_forward_hook(make_hook(i)))
    
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):
            images = images.to(device)
            _ = model(images)
            if (batch_idx + 1) % 4 == 0:
                print(f'  [{model_name}] {batch_idx+1}/{len(loader)}')
    
    for h in hooks:
        h.remove()
    
    return tracker.get_metrics()

print('Analyzer ready.')

In [ ]:
# ============================================================
# Run ViT-5-Small
# ============================================================

print('Loading ViT-5-Small...')
model_vit5 = load_vit5_small()
print(f'  {sum(p.numel() for p in model_vit5.parameters())/1e6:.1f}M params')

if device.type == 'cuda':
    print(f'  VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

print('Analyzing ViT-5-Small...')
metrics_vit5 = analyze_model(model_vit5, loader, 'ViT-5-Small')

# Extract LayerScale gammas before deleting
gamma1_vals = []
gamma2_vals = []
for block in model_vit5.blocks:
    if hasattr(block, 'gamma_1'):
        gamma1_vals.append(block.gamma_1.detach().cpu().numpy())
        gamma2_vals.append(block.gamma_2.detach().cpu().numpy())

# Free GPU
del model_vit5
torch.cuda.empty_cache()
print('ViT-5-Small done, GPU freed.')

In [ ]:
# ============================================================
# Run DeiT-III-Small
# ============================================================

print('Loading DeiT-III-Small...')
model_deit = load_deit3_small()
print(f'  {sum(p.numel() for p in model_deit.parameters())/1e6:.1f}M params')

if device.type == 'cuda':
    print(f'  VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

print('Analyzing DeiT-III-Small...')
metrics_deit = analyze_model(model_deit, loader, 'DeiT-III-Small')

del model_deit
torch.cuda.empty_cache()
print('DeiT-III-Small done, GPU freed.')

In [ ]:
# ============================================================
# Print numerical results
# ============================================================

num_layers = len(metrics_vit5)

print(f"{'Layer':>5} | {'Hoyer(V5)':>10} {'Hoyer(D3)':>10} {'diff':>7} | "
      f"{'Gini(V5)':>9} {'Gini(D3)':>9} {'diff':>7} | "
      f"{'%Zero(V5)':>10} {'%Zero(D3)':>10} | "
      f"{'%Dead(V5)':>10} {'%Dead(D3)':>10}")
print('-' * 130)

for layer in range(num_layers):
    v5 = metrics_vit5[layer]
    d3 = metrics_deit[layer]
    print(f"{layer:>5} | {v5['hoyer']:>10.4f} {d3['hoyer']:>10.4f} {v5['hoyer']-d3['hoyer']:>+7.4f} | "
          f"{v5['gini']:>9.4f} {d3['gini']:>9.4f} {v5['gini']-d3['gini']:>+7.4f} | "
          f"{v5['pct_near_zero']:>9.2f}% {d3['pct_near_zero']:>9.2f}% | "
          f"{v5['pct_dead_channels']:>9.2f}% {d3['pct_dead_channels']:>9.2f}%")

print('-' * 130)
avg_v5 = {k: np.mean([metrics_vit5[l][k] for l in range(num_layers)]) for k in ['hoyer', 'gini', 'pct_near_zero', 'pct_dead_channels']}
avg_d3 = {k: np.mean([metrics_deit[l][k] for l in range(num_layers)]) for k in ['hoyer', 'gini', 'pct_near_zero', 'pct_dead_channels']}
print(f"{'AVG':>5} | {avg_v5['hoyer']:>10.4f} {avg_d3['hoyer']:>10.4f} {avg_v5['hoyer']-avg_d3['hoyer']:>+7.4f} | "
      f"{avg_v5['gini']:>9.4f} {avg_d3['gini']:>9.4f} {avg_v5['gini']-avg_d3['gini']:>+7.4f} | "
      f"{avg_v5['pct_near_zero']:>9.2f}% {avg_d3['pct_near_zero']:>9.2f}% | "
      f"{avg_v5['pct_dead_channels']:>9.2f}% {avg_d3['pct_dead_channels']:>9.2f}%")

In [ ]:
# ============================================================
# Visualization: 4-panel comparison
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Activation Sparsity: ViT-5-Small vs DeiT-III-Small\n(Post-GELU FFN Activations)', 
             fontsize=14, fontweight='bold')

layers = list(range(num_layers))
panels = [
    ('hoyer', 'Hoyer Sparsity (higher = sparser)', 'Hoyer'),
    ('gini', 'Gini Coefficient (higher = more unequal)', 'Gini'),
    ('pct_near_zero', '% Near-Zero Activations (|x| < 1e-3)', '% Near Zero'),
    ('pct_dead_channels', '% Dead Channels (never activate)', '% Dead'),
]

for ax, (metric, title, ylabel) in zip(axes.flat, panels):
    vals_v5 = [metrics_vit5[l][metric] for l in layers]
    vals_d3 = [metrics_deit[l][metric] for l in layers]
    
    ax.plot(layers, vals_v5, 'o-', color='#e74c3c', lw=2, ms=6, label='ViT-5-Small')
    ax.plot(layers, vals_d3, 's--', color='#3498db', lw=2, ms=6, label='DeiT-III-Small')
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(layers)

plt.tight_layout()
plt.savefig('exp1_sparsity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp1_sparsity_comparison.png')

In [ ]:
# ============================================================
# Histogram of activations: single batch only (low memory)
# Reload ViT-5 briefly for 1 batch
# ============================================================

def collect_one_batch_activations(load_fn, loader):
    """Load model, run 1 batch, collect activations, free model. Peak ~200MB."""
    model = load_fn()
    activations = {}
    hooks = []
    
    for i, block in enumerate(model.blocks):
        def make_hook(idx):
            def hook_fn(module, input, output):
                # Move to CPU immediately, keep only this batch
                activations[idx] = output.detach().cpu()
            return hook_fn
        hooks.append(block.mlp.act.register_forward_hook(make_hook(i)))
    
    images, _ = next(iter(loader))
    with torch.no_grad():
        _ = model(images.to(device))
    
    for h in hooks:
        h.remove()
    del model
    torch.cuda.empty_cache()
    return activations

print('Collecting 1-batch activations for histograms...')
raw_vit5 = collect_one_batch_activations(load_vit5_small, loader)
raw_deit = collect_one_batch_activations(load_deit3_small, loader)
print('Done!')

In [ ]:
# ============================================================
# Activation histograms for layers 0, 3, 6, 11
# ============================================================

selected_layers = [0, 3, 6, 11]

fig, axes = plt.subplots(2, len(selected_layers), figsize=(18, 8))
fig.suptitle('FFN Activation Magnitude Distributions (Post-GELU, log scale)', fontsize=14, fontweight='bold')

for col, layer in enumerate(selected_layers):
    for row, (raw, name, color) in enumerate([
        (raw_vit5, 'ViT-5-Small', '#e74c3c'),
        (raw_deit, 'DeiT-III-Small', '#3498db')
    ]):
        ax = axes[row, col]
        vals = raw[layer].flatten().numpy()
        vals_clipped = np.clip(vals, -0.5, 5.0)
        ax.hist(vals_clipped, bins=200, color=color, alpha=0.8, density=True, log=True)
        ax.set_title(f'{name} — Layer {layer}', fontsize=10)
        ax.set_xlabel('Activation value')
        if col == 0:
            ax.set_ylabel('Log density')
        ax.axvspan(-0.001, 0.001, color='gray', alpha=0.3)

plt.tight_layout()
plt.savefig('exp1_activation_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

# Free histogram data
del raw_vit5, raw_deit
print('Saved: exp1_activation_histograms.png')

In [ ]:
# ============================================================
# LayerScale gamma analysis (from saved values, no GPU needed)
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('ViT-5-Small: Learned LayerScale Gamma Values', fontsize=14, fontweight='bold')

for ax, gammas, title in [
    (axes[0], gamma1_vals, 'Attention Branch (gamma_1)'),
    (axes[1], gamma2_vals, 'FFN Branch (gamma_2)')
]:
    data = np.array(gammas)
    im = ax.imshow(data, aspect='auto', cmap='RdYlBu_r', interpolation='nearest')
    ax.set_xlabel('Channel index')
    ax.set_ylabel('Layer index')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Gamma value')

plt.tight_layout()
plt.savefig('exp1_layerscale_gammas.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLayerScale gamma summary (FFN branch, gamma_2):')
for i, g in enumerate(gamma2_vals):
    print(f'  Layer {i:2d}: mean={g.mean():.6f}, std={g.std():.6f}, '
          f'min={g.min():.6f}, max={g.max():.6f}, '
          f'near-zero(<1e-4)={100*(g < 1e-4).mean():.1f}%')

In [ ]:
# ============================================================
# Effective sparsity: LayerScale gamma * FFN output
# Uses 1 batch, loads model temporarily
# ============================================================

def hoyer_sparsity(x):
    x_flat = x.reshape(x.shape[0], -1).float()
    n = x_flat.shape[1]
    l1 = x_flat.abs().sum(dim=1)
    l2 = x_flat.norm(p=2, dim=1)
    sqrt_n = np.sqrt(n)
    hoyer = (sqrt_n - l1 / (l2 + 1e-12)) / (sqrt_n - 1 + 1e-12)
    return hoyer.clamp(0, 1).mean().item()

def pct_near_zero(x, threshold=1e-3):
    return (x.abs() < threshold).float().mean().item() * 100

print('Computing effective sparsity (LayerScale * FFN output)...')
print('This shows what the residual stream actually receives.\n')

model_vit5 = load_vit5_small()

# Collect FFN output (post-MLP, before LayerScale) for 1 batch
ffn_outputs = {}
hooks = []
for i, block in enumerate(model_vit5.blocks):
    def make_hook(idx):
        def hook_fn(module, input, output):
            ffn_outputs[idx] = output.detach().cpu()
        return hook_fn
    hooks.append(block.mlp.register_forward_hook(make_hook(i)))

images, _ = next(iter(loader))
with torch.no_grad():
    _ = model_vit5(images.to(device))

for h in hooks:
    h.remove()

# Compute effective sparsity
print(f"{'Layer':>5} | {'Hoyer(raw)':>11} {'Hoyer(scaled)':>14} {'amplif.':>8} | "
      f"{'%Zero(raw)':>11} {'%Zero(scaled)':>14}")
print('-' * 75)

eff_hoyer_raw = []
eff_hoyer_scaled = []

for i, block in enumerate(model_vit5.blocks):
    if hasattr(block, 'gamma_2'):
        gamma = block.gamma_2.detach().cpu()
        raw = ffn_outputs[i]
        scaled = raw * gamma
        
        h_raw = hoyer_sparsity(raw)
        h_scaled = hoyer_sparsity(scaled)
        z_raw = pct_near_zero(raw)
        z_scaled = pct_near_zero(scaled)
        
        eff_hoyer_raw.append(h_raw)
        eff_hoyer_scaled.append(h_scaled)
        
        print(f"{i:>5} | {h_raw:>11.4f} {h_scaled:>14.4f} {h_scaled-h_raw:>+8.4f} | "
              f"{z_raw:>10.2f}% {z_scaled:>13.2f}%")

print('-' * 75)
print(f"{'AVG':>5} | {np.mean(eff_hoyer_raw):>11.4f} {np.mean(eff_hoyer_scaled):>14.4f} "
      f"{np.mean(eff_hoyer_scaled)-np.mean(eff_hoyer_raw):>+8.4f} |")

del model_vit5, ffn_outputs
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Summary and interpretation
# ============================================================

print('=' * 70)
print('SUMMARY: Activation Sparsity Analysis')
print('=' * 70)
print()

print('Q1: Does ViT-5 have different sparsity than DeiT-III?')
print(f'    ViT-5   avg Hoyer: {avg_v5["hoyer"]:.4f}')
print(f'    DeiT-III avg Hoyer: {avg_d3["hoyer"]:.4f}')
diff = avg_v5['hoyer'] - avg_d3['hoyer']
print(f'    Delta: {diff:+.4f} ({"ViT-5 sparser" if diff > 0 else "DeiT-III sparser"})')
print()

print('Q2: Does LayerScale suppress channels ("static gating")?')
all_g = np.concatenate(gamma2_vals)
print(f'    FFN gamma mean={all_g.mean():.6f}, min={all_g.min():.6f}')
print(f'    Channels with gamma < 1e-4: {100*(all_g < 1e-4).mean():.1f}%')
print(f'    Channels with gamma < 1e-3: {100*(all_g < 1e-3).mean():.1f}%')
print()

print('Q3: Does LayerScale amplify sparsity of the residual signal?')
print(f'    Avg Hoyer raw FFN:       {np.mean(eff_hoyer_raw):.4f}')
print(f'    Avg Hoyer after scaling:  {np.mean(eff_hoyer_scaled):.4f}')
print(f'    Amplification: {np.mean(eff_hoyer_scaled)-np.mean(eff_hoyer_raw):+.4f}')
print()

print('INTERPRETATION:')
print('- If ViT-5 shows HIGHER sparsity than DeiT-III, LayerScale acts as')
print('  "static gating" => adding SwiGLU would compound it ("over-gating").')
print('- If sparsity is SIMILAR, the over-gating explanation is weak.')
print('- If ViT-5 shows LOWER sparsity, the narrative is wrong.')
print()
print('NOTE: The paper never measures any of these quantities.')
print('Their "over-gating" claim is purely speculative without this data.')